# University GPU server edition

Run cells from top to bottom in a **fresh kernel for each notebook**, inside a GPU allocation.
See `README.md` in this folder for setup and known limitations. This copy does not install
packages, replace PyTorch, mount Drive, or modify the original checkpoint directory.
Outputs go to `GPU folders/outputs` (override with `DOOM_OUTPUT_ROOT` before starting Jupyter).
Training settings and evaluation episode counts retain the original defaults: these are full runs.
For direct SNN training, set `SMOKE_TEST = True` first for a short training check.


In [ ]:
import os, sys, json, platform
from pathlib import Path
from importlib import metadata

# Works when Jupyter starts in either the repository or GPU folders.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "doom_plaincnn_sanity_check.ipynb").is_file()
                  and (p / "GPU folders").is_dir()
                  and (p / "conversion_friendly_v1").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the doom-on-snn repository or GPU folders.")
WORK_DIR = REPO_ROOT / "GPU folders"
OUTPUT_ROOT = Path(os.environ.get("DOOM_OUTPUT_ROOT", str(WORK_DIR / "outputs"))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
# Match worker count to the CPU allocation; allow an explicit lower limit.
CPU_BUDGET = int(os.environ.get("SLURM_CPUS_PER_TASK", len(os.sched_getaffinity(0))))
SERVER_WORKERS = max(1, min(8, int(os.environ.get("DOOM_NUM_WORKERS", max(1, CPU_BUDGET - 2))), CPU_BUDGET))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import numpy as np
import torch
import vizdoom, sample_factory
torch.set_num_threads(1)
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU visible. Run the Jupyter kernel inside your cluster GPU allocation.")
print("Python:", sys.version, "\nKernel:", sys.executable)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU calculation:", (torch.ones(1, device="cuda") + 1).item())
print("CPU budget:", CPU_BUDGET, "workers:", SERVER_WORKERS)
print("Repository:", REPO_ROOT, "\nOutputs:", OUTPUT_ROOT)
versions = {p: metadata.version(p) for p in
            ["torch", "numpy", "sample-factory", "vizdoom", "gymnasium", "matplotlib"]}
(OUTPUT_ROOT / "environment.json").write_text(json.dumps({
    "python": sys.version, "executable": sys.executable, "packages": versions,
    "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
}, indent=2))


# DOOM Sample Factory Pipeline (v4)

Changes from v3:
- Fixed Section 7 (the encoder-inspection cell). `make_doom_env_impl` and `create_env`
  both return a plain `Box` observation space, but `VizdoomEncoder` expects a `Dict`
  with an `"obs"` key -- that dict-wrapping happens deeper inside Sample Factory's model
  construction, not at the environment level. We now build the expected dict shape
  manually before calling `make_vizdoom_encoder`.

**Run on the university Linux GPU server.**
Start the kernel inside your GPU allocation.

### 1. Use the existing server environment

Dependencies are checked in the first code cell. No administrator commands are run.


In [ ]:
# Dependencies are checked in the server setup cell. No installs or CUDA replacements here.


### 2. Dependency check

Already performed in the first code cell.


In [ ]:
# Dependencies are checked in the server setup cell. No installs or CUDA replacements here.


### 3. Keep the existing CUDA PyTorch build

The server setup checks a real GPU calculation without reinstalling PyTorch.


In [ ]:
# Dependencies are checked in the server setup cell. No installs or CUDA replacements here.


Continue in the same kernel. No packages were installed by this notebook.


In [ ]:
print("Using existing CUDA PyTorch:", torch.__version__)


### 4. The PyTorch 2.6 checkpoint-loading fix
Forces `torch.load` back to its pre-2.6 permissive default. Safe here because we only
ever load checkpoints trained by us, in this same notebook.

In [ ]:
# Sample Factory also calls torch.load internally. Its trusted training checkpoints
# contain NumPy metadata. Preserve explicit weights_only arguments; patch only once.
# Only use checkpoints you produced or obtained from a trusted source.
if not getattr(torch.load, "_doom_trusted_load", False):
    def _trusted_load(*args, _load=torch.load, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _load(*args, **kwargs)
    _trusted_load._doom_trusted_load = True
    torch.load = _trusted_load
print("Trusted checkpoint loading enabled; models and inputs still run on CUDA.")


### 5. Register Sample Factory's ViZDoom environments and models

In [ ]:
import functools

from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.envs.create_env import create_env
from sample_factory.train import run_rl

from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec


def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        make_env_func = functools.partial(make_doom_env_from_spec, env_spec)
        register_env(env_spec.name, make_env_func)


def register_vizdoom_models():
    """Registers Sample Factory's default ViZDoom-tuned CNN encoder.

    NOTE: default uses ELU and a GRU -- both overridden via config flags below
    (--nonlinearity=relu, --use_rnn=False) rather than needing a fully custom encoder,
    as long as BatchNorm is not also present (checked in Section 7).
    """
    global_model_factory().register_encoder_factory(make_vizdoom_encoder)


def register_vizdoom_components():
    register_vizdoom_envs()
    register_vizdoom_models()


def parse_vizdoom_cfg(argv=None, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser)
    doom_override_defaults(parser)
    final_cfg = parse_full_cfg(parser, argv)
    return final_cfg


register_vizdoom_components()

### 6. Choose the scenario and training config

`--nonlinearity=relu`: overrides the ELU default -- required for ANN-to-SNN conversion
compatibility.

`--use_rnn=False` + `--env_framestack=4`: use frame-stacking for temporal info instead of
a GRU -- keeps the architecture a plain feedforward CNN, avoiding the much harder
recurrent-SNN conversion problem.

In [ ]:
env = "doom_basic"
train_for_env_steps = 2000000   # raised from 500000 to resume training
num_workers = SERVER_WORKERS
num_envs_per_worker = 4

conversion_friendly_args = [
    f"--env={env}",
    "--experiment=conversion_friendly_v1",
    f"--num_workers={num_workers}",
    f"--num_envs_per_worker={num_envs_per_worker}",
    f"--train_for_env_steps={train_for_env_steps}",
    "--nonlinearity=relu",
    "--use_rnn=False",
]
TRAIN_DIR = str(OUTPUT_ROOT / "train_dir")
conversion_friendly_args += [f"--train_dir={TRAIN_DIR}", "--device=gpu", "--set_workers_cpu_affinity=False"]


### 7. Inspect the encoder BEFORE training

`create_env` returns a plain `Box` observation space, but `VizdoomEncoder` (the class
wrapping the actual CNN) expects a `Dict` with an `"obs"` key -- that wrapping normally
happens deeper inside Sample Factory's model-construction code, not at the environment
level. We build the expected dict shape manually here so we can inspect the encoder in
isolation.

Check the printed output for any `BatchNorm` layer -- if one appears, it needs to be
folded into the preceding conv layer after training (or replaced with a custom encoder)
before this model can be converted to an SNN.

In [ ]:
inspect_cfg = parse_vizdoom_cfg(argv=conversion_friendly_args)
inspect_env = create_env(env, cfg=inspect_cfg, env_config=None)

# VizdoomEncoder expects obs_space["obs"] (and optionally obs_space["measurements"]),
# but create_env returns a plain Box, not a Dict -- build the dict shape it expects.
obs_space_for_encoder = {"obs": inspect_env.observation_space}

inspect_encoder = make_vizdoom_encoder(inspect_cfg, obs_space_for_encoder)

print(inspect_encoder)

has_batchnorm = any("BatchNorm" in type(m).__name__ for m in inspect_encoder.modules())
print("\nContains BatchNorm:", has_batchnorm)

inspect_env.close()

### 8. Train
Only proceed once Section 7 confirms `Contains BatchNorm: False` -- otherwise, resolve
that first (see the closing notes).

In [ ]:
cfg = parse_vizdoom_cfg(argv=conversion_friendly_args)
status = run_rl(cfg)

### 9. Confirm training actually used the GPU and the correct overrides

In [ ]:
import json

with open(Path(TRAIN_DIR) / "conversion_friendly_v1" / "config.json") as f:
    saved_cfg = json.load(f)

print("Device used for training:", saved_cfg.get("device"))
print("Nonlinearity used:", saved_cfg.get("nonlinearity"))
print("RNN used:", saved_cfg.get("use_rnn"))

### 10. Evaluate (50 episodes)

In [ ]:
import re, logging, numpy as np
from sample_factory.enjoy import enjoy

class Grab(logging.Handler):
    def __init__(self):
        super().__init__()
        self.avgs = []
    def emit(self, record):
        m = re.search(r"Avg episode reward: ([-\d.]+),", record.getMessage())
        if m:
            self.avgs.append(float(m.group(1)))

grab = Grab()
logging.getLogger("rl").addHandler(grab)   # assumes SF logs under "rl"

eval_cfg = parse_vizdoom_cfg(
    argv=[
        f"--env={env}",
        "--experiment=conversion_friendly_v1",
        "--num_workers=1",
        f"--train_dir={TRAIN_DIR}",
        "--device=gpu",
        "--no_render",
        "--max_num_episodes=50",
        "--eval_deterministic=True",
        # Video is optional; enable --save_video only if ffmpeg is available.
    ],
    evaluation=True,
)
enjoy(eval_cfg)

a = np.array(grab.avgs)
n = np.arange(1, len(a) + 1)
eps = n * a - (n - 1) * np.concatenate([[0], a[:-1]])
print("episodes:", len(eps))
print(f"mean {eps.mean():.3f} ± {eps.std():.3f}")
print("episodes below 0:", (eps < 0).sum())
print("worst 5:", np.sort(eps)[:5].round(2))

### 11. Watch the replay

In [ ]:
from IPython.display import Video, display
video_path = Path(TRAIN_DIR) / "conversion_friendly_v1" / "replay.mp4"
if video_path.is_file():
    display(Video(str(video_path), embed=True))
else:
    print("No video saved. Enable --save_video in the evaluation cell if ffmpeg is installed.")


### 12. Record your results here

- Scenario: `doom_basic`
- Total training steps: 2,000,000
- nonlinearity: relu | use_rnn: False | single frame (no frame stacking).
- Contains BatchNorm: False
- Device used: GPU
- Evaluation episodes: 50
- Mean reward ± std: (fill in from `enjoy()`'s printed output)
- Checkpoint path: `train_dir/conversion_friendly_v1/checkpoint_p0/...`
- Date run: (fill in)

### Notes for the project going forward

- **Official Phase 1 ANN baseline pipeline**, replacing both the earlier SB3 notebook
  (unresolved evaluation discrepancy) and the v1/v2 Sample Factory notebooks (used
  default ELU + GRU, not conversion-friendly).
- **If Section 7 shows BatchNorm present**: either (a) fold BatchNorm into the preceding
  conv layer's weights after training (`torch.nn.utils.fuse_conv_bn_eval` can help), or
  (b) write and register a fully custom encoder matching `ConversionFriendlyCNN`'s
  constraints, via `global_model_factory().register_encoder_factory(...)`.
- **Reproducibility**: retain the checkpoint, `config.json`, and environment report under the persistent output directory.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_ROOT / "conversion_friendly_v1_backup"), "zip",
                              Path(TRAIN_DIR) / "conversion_friendly_v1")
print("Backup:", archive)


In [ ]:
print("Training and backup files persist under", OUTPUT_ROOT)


In [ ]:
# Colab package-source download diagnostic omitted.
